# MediaPipe Extraction — INCLUDE-50 reference benchmark split (v4)

This notebook rebuilds the 50-class INCLUDE benchmark input and uses the **recovered reference train/validation/test split** employed by the Dudu/Alves experimental pipeline.

Important distinction:

- the INCLUDE-50 benchmark contains 50 signs and 958 referenced videos;
- the recovered protocol uses 689 training, 77 validation, and 192 test references;
- the original 766-video training pool is therefore `689 + 77`;
- the three split files are downloaded from a **commit-pinned public mirror** only when the local files are missing or empty;
- no `train_test_split` or random fallback is permitted;
- every split reference must resolve to exactly one raw video before MediaPipe extraction starts.

If a reference is absent from the local raw archive, the notebook stops and prints the exact missing path instead of silently changing the benchmark.


### v4 fix

`sequence_id` is now derived from a SHA-256 digest of the normalized relative video path. This prevents filename collisions such as `MVI_0060.MOV` versus `MVI_0060_.MOV`, both of which legitimately exist in the raw INCLUDE archive.


In [1]:

from pathlib import Path
import re
import hashlib
import warnings
import urllib.request
import json
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
from difflib import get_close_matches

import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

# Expected usage: notebook is executed from <PROJECT_ROOT>/notebooks
PROJECT_ROOT = Path.cwd().parent

RAW_ROOT = PROJECT_ROOT / "data/raw/include50"

# IMPORTANT: use a NEW output directory; do not overwrite the old 929-video data.
OUTPUT_DIR = PROJECT_ROOT / "data/interim/include50_official"
SHARDS_DIR = OUTPUT_DIR / "shards"

MANIFEST_PATH = OUTPUT_DIR / "include50_official_manifest.csv"
CLASS_INVENTORY_PATH = OUTPUT_DIR / "include50_official_class_inventory.csv"
FINAL_CSV_PATH = OUTPUT_DIR / "include50_official_mediapipe_with_split.csv"
PROVENANCE_PATH = OUTPUT_DIR / "include50_reference_split_provenance.json"

# -------------------------------------------------------------------------
# Recovered reference partition files
# -------------------------------------------------------------------------
# Dudu's repository reads these three filenames but does not version them.
# A public mirror containing the same 689/77/192 lists was recovered and is
# pinned below to an immutable Git commit.
#
# Existing NON-EMPTY local files are never overwritten automatically.
# Empty placeholder files are replaced by the pinned reference copies.
TRAIN_SPLIT_FILE = RAW_ROOT / "include50_train.txt"
VAL_SPLIT_FILE = RAW_ROOT / "include50_val.txt"
TEST_SPLIT_FILE = RAW_ROOT / "include50_test.txt"

AUTO_DOWNLOAD_REFERENCE_SPLITS = True

REFERENCE_SPLIT_REPOSITORY = "Nittaany/Major-Project"
REFERENCE_SPLIT_COMMIT = "053ef098718fa5e9600a00508e1968e6b94bfdfd"
REFERENCE_SPLIT_BASE_URL = (
    "https://raw.githubusercontent.com/"
    f"{REFERENCE_SPLIT_REPOSITORY}/{REFERENCE_SPLIT_COMMIT}/src/ml_tools"
)

REFERENCE_SPLIT_URLS = {
    "train": f"{REFERENCE_SPLIT_BASE_URL}/include50_train.txt",
    "val": f"{REFERENCE_SPLIT_BASE_URL}/include50_val.txt",
    "test": f"{REFERENCE_SPLIT_BASE_URL}/include50_test.txt",
}

EXPECTED_SPLIT_COUNTS = {
    "train": 689,
    "val": 77,
    "test": 192,
}

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv", ".MOV", ".MP4", ".AVI", ".MKV"}

# Extraction controls
MAX_VIDEOS = None      # Use e.g. 5 only for a quick extraction test.
RESUME = True

# Dataset invariants
EXPECTED_N_CLASSES = 50
EXPECTED_N_VIDEOS = 958
EXPECTED_OFFICIAL_TRAIN_POOL = 766   # train + validation
EXPECTED_TRAIN = EXPECTED_SPLIT_COUNTS["train"]
EXPECTED_VAL = EXPECTED_SPLIT_COUNTS["val"]
EXPECTED_TEST = EXPECTED_SPLIT_COUNTS["test"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TRAIN_SPLIT_FILE:", TRAIN_SPLIT_FILE)
print("VAL_SPLIT_FILE:", VAL_SPLIT_FILE)
print("TEST_SPLIT_FILE:", TEST_SPLIT_FILE)

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Raw INCLUDE directory not found: {RAW_ROOT}")


PROJECT_ROOT: /Users/dani/Projetos/islr-subset
RAW_ROOT: /Users/dani/Projetos/islr-subset/data/raw/include50
OUTPUT_DIR: /Users/dani/Projetos/islr-subset/data/interim/include50_official
TRAIN_SPLIT_FILE: /Users/dani/Projetos/islr-subset/data/raw/include50/include50_train.txt
VAL_SPLIT_FILE: /Users/dani/Projetos/islr-subset/data/raw/include50/include50_val.txt
TEST_SPLIT_FILE: /Users/dani/Projetos/islr-subset/data/raw/include50/include50_test.txt



## Official INCLUDE-50 vocabulary

The class list below is fixed. The notebook never replaces it with a data-driven selection.

Matching is case-insensitive and punctuation-insensitive (for example, `T-Shirt` and `T shirt` normalize to the same label), but each official label must resolve to **exactly one** class in the raw dataset.


In [2]:

OFFICIAL_INCLUDE50_SIGNS = [
    "Bank",
    "Bird",
    "Black",
    "Boy",
    "Brother",
    "Car",
    "Cell phone",
    "Court",
    "Cow",
    "Death",
    "Dog",
    "Election",
    "Fall",
    "Fan",
    "Father",
    "Girl",
    "Good Morning",
    "Hat",
    "Hello",
    "House",
    "I",
    "Monday",
    "Paint",
    "Pen",
    "Priest",
    "Red",
    "Shoes",
    "Shop",
    "Summer",
    "T-Shirt",
    "Teacher",
    "Thank you",
    "Time",
    "White",
    "Window",
    "Year",
    "Large",
    "Dry",
    "Good",
    "Happy",
    "Hot",
    "It",
    "Long",
    "Loud",
    "New",
    "Quiet",
    "Short",
    "Small",
    "Train Ticket",
    "You (plural)",
]

assert len(OFFICIAL_INCLUDE50_SIGNS) == EXPECTED_N_CLASSES
assert len(set(OFFICIAL_INCLUDE50_SIGNS)) == EXPECTED_N_CLASSES

print("Official classes:", len(OFFICIAL_INCLUDE50_SIGNS))


Official classes: 50


In [3]:
def norm_text(value) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def norm_sign(value) -> str:
    # Stronger normalization only for class-name matching.
    value = str(value).strip().lower()
    value = value.replace("-", " ")
    value = re.sub(r"[()]", " ", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def norm_video_ref(value) -> str:
    value = str(value).strip().replace("\\", "/").lower()
    value = re.sub(r"^\./", "", value)
    value = re.sub(r"\.(mov|mp4|avi|mkv)$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"/+", "/", value)
    return value.strip("/")


def slugify(value: str) -> str:
    value = str(value)
    value = value.replace("/", "__").replace("\\", "__")
    value = re.sub(r"[^A-Za-z0-9_\-.]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def parse_class_folder(folder_name: str):
    """
    Parse a regular INCLUDE class folder.

    Examples
    --------
    "1. loud" -> (1, "loud", False)
    "84. small little-extra" -> (84, "small little", True)

    NOTE:
    Most extra samples in the raw archive are stored in a nested
    `Extra/` directory. That case is detected separately while
    traversing the path. This function also supports a suffix form
    as a defensive fallback.
    """
    m = re.match(r"^\s*(\d+(?:\.\d+)?)\.\s*(.+?)\s*$", folder_name)

    if not m:
        sign_raw = folder_name.strip()
        suffix_extra = bool(
            re.search(r"\s*-\s*extra\s*$", sign_raw, flags=re.IGNORECASE)
        )
        sign = re.sub(
            r"\s*-\s*extra\s*$",
            "",
            sign_raw,
            flags=re.IGNORECASE,
        ).strip()
        return np.nan, sign, suffix_extra

    raw_id = m.group(1)
    sign_raw = m.group(2).strip()

    try:
        original_class_id = int(float(raw_id))
    except Exception:
        original_class_id = np.nan

    suffix_extra = bool(
        re.search(r"\s*-\s*extra\s*$", sign_raw, flags=re.IGNORECASE)
    )

    sign = re.sub(
        r"\s*-\s*extra\s*$",
        "",
        sign_raw,
        flags=re.IGNORECASE,
    ).strip()

    return original_class_id, sign, suffix_extra


def format_original_key(original_class_id, category, sign):
    if pd.isna(original_class_id):
        return f"{category}::{sign}"
    return f"{int(original_class_id):02d}::{category}::{sign}"

## Discover the raw INCLUDE videos

In [4]:
video_paths = sorted(
    p for p in RAW_ROOT.rglob("*")
    if p.is_file() and p.suffix in VIDEO_EXTENSIONS
)

records = []

for video_path in video_paths:
    rel = video_path.relative_to(RAW_ROOT)
    parts = rel.parts

    # Normal layout:
    #   package / category / class_folder / video
    #
    # Extra-sample layout used by INCLUDE:
    #   package / category / class_folder / Extra / video
    #
    # The previous notebook incorrectly interpreted `Extra` as the
    # class folder and shifted the category one level down.
    nested_extra = (
        len(parts) >= 5
        and parts[-2].strip().lower() == "extra"
    )

    if nested_extra:
        package = parts[-5]
        category = parts[-4]
        class_folder = parts[-3]
    else:
        if len(parts) < 4:
            warnings.warn(f"Skipping path with unexpected structure: {rel}")
            continue

        package = parts[-4]
        category = parts[-3]
        class_folder = parts[-2]

    video_name = parts[-1]

    original_class_id, sign, suffix_extra = parse_class_folder(class_folder)
    is_extra = bool(nested_extra or suffix_extra)

    class_key = f"{category}::{sign}"
    original_key = format_original_key(original_class_id, category, sign)
    # IMPORTANT: use a collision-proof identifier.
    # The previous slugified ID collapsed paths such as MVI_0060.MOV and
    # MVI_0060_.MOV to the same sequence_id.
    video_ref = norm_video_ref(rel.as_posix())
    sequence_id = hashlib.sha256(
        video_ref.encode("utf-8")
    ).hexdigest()[:24]

    records.append({
        "video_path": str(video_path),
        "video_relpath": rel.as_posix(),
        "package": package,
        "category": category,
        "class_folder": class_folder,
        "original_class_id": original_class_id,
        "sign": sign,
        "sign_norm": norm_sign(sign),
        "class_key": class_key,
        "original_key": original_key,
        "video_name": video_name,
        "sequence_id": sequence_id,
        "video_ref": video_ref,
        "video_basename_ref": norm_video_ref(video_name),
        "is_extra": is_extra,
    })

raw_metadata = pd.DataFrame(records)

duplicate_sequence_ids = raw_metadata[
    raw_metadata.duplicated("sequence_id", keep=False)
].copy()

if len(duplicate_sequence_ids):
    display(
        duplicate_sequence_ids[
            ["sequence_id", "video_relpath", "video_ref"]
        ].sort_values("sequence_id")
    )
    raise RuntimeError(
        "sequence_id collision detected. Each raw video must have a unique identifier."
    )

print("Raw videos discovered:", len(video_paths))
print("Videos with valid metadata:", len(raw_metadata))
print("Raw classes after Extra-folder normalization:", raw_metadata["class_key"].nunique())
print("Videos marked as Extra:", int(raw_metadata["is_extra"].sum()))

display(raw_metadata.head())

Raw videos discovered: 4284
Videos with valid metadata: 4284
Raw classes after Extra-folder normalization: 262
Videos marked as Extra: 27


,video_path,video_relpath,package,category,class_folder,original_class_id,sign,sign_norm,class_key,original_key,video_name,sequence_id,video_ref,video_basename_ref,is_extra
0,/Users/dani/Projetos/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5177.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,loud,Adjectives::loud,01::Adjectives::loud,MVI_5177.MOV,8e924f942511649db1e74819,adjectives_1of8/adjectives/1. loud/mvi_5177,mvi_5177,False
1,/Users/dani/Projetos/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5178.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,loud,Adjectives::loud,01::Adjectives::loud,MVI_5178.MOV,646726495be790c8c770f390,adjectives_1of8/adjectives/1. loud/mvi_5178,mvi_5178,False
2,/Users/dani/Projetos/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5179.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,loud,Adjectives::loud,01::Adjectives::loud,MVI_5179.MOV,071283f88741f4d863825d84,adjectives_1of8/adjectives/1. loud/mvi_5179,mvi_5179,False
3,/Users/dani/Projetos/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5257.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,loud,Adjectives::loud,01::Adjectives::loud,MVI_5257.MOV,266921c81e09849c6b72b789,adjectives_1of8/adjectives/1. loud/mvi_5257,mvi_5257,False
4,/Users/dani/Projetos/islr-subset/data/raw/incl...,Adjectives_1of8/Adjectives/1. loud/MVI_5258.MOV,Adjectives_1of8,Adjectives,1. loud,1.0,loud,loud,Adjectives::loud,01::Adjectives::loud,MVI_5258.MOV,0c547329c17ca52829c3ce27,adjectives_1of8/adjectives/1. loud/mvi_5258,mvi_5258,False


In [5]:

# Inventory of all raw classes
classes_df = (
    raw_metadata
    .groupby(
        ["class_key", "original_key", "category", "sign", "sign_norm", "original_class_id"],
        dropna=False,
    )
    .agg(
        n_videos=("sequence_id", "nunique"),
        first_video=("video_relpath", "first"),
    )
    .reset_index()
    .sort_values(["category", "original_class_id", "sign"], na_position="last")
    .reset_index(drop=True)
)

display(classes_df)


,class_key,original_key,category,sign,sign_norm,original_class_id,n_videos,first_video
0,Adjectives::loud,01::Adjectives::loud,Adjectives,loud,loud,1.0,21,Adjectives_1of8/Adjectives/1. loud/MVI_5177.MOV
1,Adjectives::quiet,02::Adjectives::quiet,Adjectives,quiet,quiet,2.0,21,Adjectives_1of8/Adjectives/2. quiet/MVI_5180.MOV
2,Adjectives::happy,03::Adjectives::happy,Adjectives,happy,happy,3.0,21,Adjectives_1of8/Adjectives/3. happy/MVI_5183.MOV
3,Adjectives::sad,04::Adjectives::sad,Adjectives,sad,sad,4.0,8,Adjectives_1of8/Adjectives/4. sad/MVI_9565.MOV
4,Adjectives::Beautiful,05::Adjectives::Beautiful,Adjectives,Beautiful,beautiful,5.0,9,Adjectives_1of8/Adjectives/5. Beautiful/Extra/...
...,...,...,...,...,...,...,...,...
257,Society::Ball,19::Society::Ball,Society,Ball,ball,19.0,15,Society_3of3/Society/19. Ball/MVI_4324.MOV
258,Society::Price,20::Society::Price,Society,Price,price,20.0,14,Society_3of3/Society/20. Price/MVI_4328.MOV
259,Society::Sign,21::Society::Sign,Society,Sign,sign,21.0,14,Society_3of3/Society/21. Sign/MVI_4332.MOV
260,Society::Science,22::Society::Science,Society,Science,science,22.0,14,Society_3of3/Society/22. Science/MVI_4336.MOV



## Resolve the official 50 classes

This step is deliberately strict:

- zero matches → stop;
- more than one raw class matching an official label → stop;
- exactly one match → accept.

No replacement class is chosen automatically.


In [6]:
# The benchmark labels and the raw folder labels differ for three classes.
# Keep the benchmark names in `official_sign`, and use aliases only for
# resolving the raw folders.
OFFICIAL_SIGN_ALIASES = {
    "Shop": "Store or Shop",
    "Large": "big large",
    "Small": "small little",
}

resolved_rows = []
problems = []

available_norms = sorted(classes_df["sign_norm"].dropna().unique())

for official_order, official_sign in enumerate(OFFICIAL_INCLUDE50_SIGNS):
    raw_sign = OFFICIAL_SIGN_ALIASES.get(official_sign, official_sign)
    key = norm_sign(raw_sign)

    matches = classes_df[classes_df["sign_norm"] == key].copy()

    if len(matches) == 1:
        row = matches.iloc[0].to_dict()
        row["official_order"] = official_order
        row["official_sign"] = official_sign
        row["raw_sign_alias"] = raw_sign
        resolved_rows.append(row)
        continue

    if len(matches) == 0:
        close = get_close_matches(key, available_norms, n=8, cutoff=0.45)
        problems.append({
            "official_sign": official_sign,
            "raw_sign_alias": raw_sign,
            "problem": "not found",
            "candidates": close,
        })
    else:
        problems.append({
            "official_sign": official_sign,
            "raw_sign_alias": raw_sign,
            "problem": f"ambiguous ({len(matches)} matches)",
            "candidates": matches["class_key"].tolist(),
        })

if problems:
    problems_df = pd.DataFrame(problems)
    display(problems_df)
    raise RuntimeError(
        "The official INCLUDE-50 vocabulary could not be resolved uniquely. "
        "Review the displayed labels/candidates before continuing."
    )

selected_classes = (
    pd.DataFrame(resolved_rows)
    .sort_values("official_order")
    .reset_index(drop=True)
)

selected_classes["sign_id"] = np.arange(len(selected_classes), dtype=int)

if selected_classes["class_key"].nunique() != EXPECTED_N_CLASSES:
    raise RuntimeError("The 50 official labels did not resolve to 50 distinct raw classes.")

selected_classes.to_csv(CLASS_INVENTORY_PATH, index=False)

print("Resolved official classes:", len(selected_classes))
display(
    selected_classes[
        ["sign_id", "official_sign", "raw_sign_alias",
         "category", "sign", "class_key", "n_videos"]
    ]
)

Resolved official classes: 50


,sign_id,official_sign,raw_sign_alias,category,sign,class_key,n_videos
0,0,Bank,Bank,Places,Bank,Places::Bank,22
1,1,Bird,Bird,Animals,Bird,Animals::Bird,25
2,2,Black,Black,Colours,Black,Colours::Black,20
3,3,Boy,Boy,People,Boy,People::Boy,21
4,4,Brother,Brother,People,Brother,People::Brother,21
5,5,Car,Car,Means_of_Transportation,Car,Means_of_Transportation::Car,20
6,6,Cell phone,Cell phone,Electronics,Cell phone,Electronics::Cell phone,14
7,7,Court,Court,Places,Court,Places::Court,23
8,8,Cow,Cow,Animals,Cow,Animals::Cow,21
9,9,Death,Death,Society,Death,Society::Death,14


## Build the 50-class raw candidate pool

At this stage the notebook verifies the **50 official classes**, but it does
not assume that every raw video from those folders belongs to the benchmark.

Exact benchmark membership is determined in the next step by the reference
`train`, `val`, and `test` files. This is important because the raw archive
contains nested `Extra/` directories and may contain a slightly different
set of files from a particular benchmark release.

In [7]:
class_to_sign_id = dict(zip(selected_classes["class_key"], selected_classes["sign_id"]))
class_to_official_sign = dict(zip(selected_classes["class_key"], selected_classes["official_sign"]))

metadata = raw_metadata[raw_metadata["class_key"].isin(class_to_sign_id)].copy()

metadata["sign_id"] = metadata["class_key"].map(class_to_sign_id).astype(int)
metadata["official_sign"] = metadata["class_key"].map(class_to_official_sign)

metadata = metadata.sort_values(
    ["sign_id", "video_relpath"]
).reset_index(drop=True)

metadata["sample_id"] = metadata.groupby("sign_id").cumcount()

n_classes = metadata["sign_id"].nunique()
n_candidate_videos = metadata["sequence_id"].nunique()

print("Resolved classes:", n_classes)
print("Candidate videos in the raw data:", n_candidate_videos)

per_class = (
    metadata.groupby(["sign_id", "official_sign", "class_key"])["sequence_id"]
    .nunique()
    .rename("n_videos")
    .reset_index()
)

display(per_class)

if n_classes != EXPECTED_N_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_N_CLASSES} classes, found {n_classes}."
    )

if n_candidate_videos != EXPECTED_N_VIDEOS:
    print(
        f"\nNOTE: the raw candidate pool contains {n_candidate_videos} videos, "
        f"not {EXPECTED_N_VIDEOS}. This is NOT used to redefine the benchmark. "
        "The reference train/val/test files are checked next and determine "
        "the exact 958 benchmark members."
    )

print("\nProceeding to strict matching against the reference split files.")

Resolved classes: 50
Candidate videos in the raw data: 958


,sign_id,official_sign,class_key,n_videos
0,0,Bank,Places::Bank,22
1,1,Bird,Animals::Bird,25
2,2,Black,Colours::Black,20
3,3,Boy,People::Boy,21
4,4,Brother,People::Brother,21
5,5,Car,Means_of_Transportation::Car,20
6,6,Cell phone,Electronics::Cell phone,14
7,7,Court,Places::Court,23
8,8,Cow,Animals::Cow,21
9,9,Death,Society::Death,14



Proceeding to strict matching against the reference split files.


## Load the recovered reference train/validation/test files

The notebook does **not** create a new random split.

The local files are expected at:

- `data/raw/include50/include50_train.txt`
- `data/raw/include50/include50_val.txt`
- `data/raw/include50/include50_test.txt`

If one of these files is missing or has size zero, v3 downloads the corresponding file from the commit-pinned recovered mirror. A non-empty local file with an unexpected number of references is **not overwritten**; the notebook stops so that a potentially different protocol cannot be silently replaced.


In [8]:

required_split_files = {
    "train": TRAIN_SPLIT_FILE,
    "val": VAL_SPLIT_FILE,
    "test": TEST_SPLIT_FILE,
}


def read_split_file(path: Path) -> List[str]:
    lines = []
    with path.open("r", encoding="utf-8-sig") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            lines.append(line)
    return lines


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def download_reference_split(split: str, path: Path) -> None:
    url = REFERENCE_SPLIT_URLS[split]
    expected = EXPECTED_SPLIT_COUNTS[split]

    path.parent.mkdir(parents=True, exist_ok=True)

    # Never silently replace a non-empty file.
    if path.exists() and path.stat().st_size > 0:
        current = read_split_file(path)
        if len(current) != expected:
            raise RuntimeError(
                f"{path} is non-empty but contains {len(current)} references; "
                f"the recovered reference split expects {expected}. "
                "The notebook will not overwrite it automatically."
            )
        print(f"{split}: using existing local file ({len(current)} references)")
        return

    if not AUTO_DOWNLOAD_REFERENCE_SPLITS:
        raise FileNotFoundError(
            f"{path} is missing or empty and automatic download is disabled."
        )

    tmp = path.with_suffix(path.suffix + ".download")
    if tmp.exists():
        tmp.unlink()

    print(f"{split}: downloading recovered reference split...")
    print("  source:", url)

    try:
        urllib.request.urlretrieve(url, tmp)
    except Exception as exc:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(
            f"Failed to download the {split} reference split from the pinned "
            f"GitHub commit. Download it manually and save it as {path}."
        ) from exc

    downloaded = read_split_file(tmp)
    if len(downloaded) != expected:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(
            f"Downloaded {split} split contains {len(downloaded)} references; "
            f"expected {expected}. Refusing to install it."
        )

    tmp.replace(path)
    print(f"  installed: {path} ({expected} references)")


for split, path in required_split_files.items():
    download_reference_split(split, path)


split_entries = {
    split: read_split_file(path)
    for split, path in required_split_files.items()
}

print("\nRecovered reference split:")
for split, entries in split_entries.items():
    expected = EXPECTED_SPLIT_COUNTS[split]
    print(
        f"{split}: {len(entries)} references "
        f"(expected {expected}) | sha256={sha256_file(required_split_files[split])}"
    )
    if len(entries) != expected:
        raise RuntimeError(
            f"{split} contains {len(entries)} references; expected {expected}."
        )

if sum(len(v) for v in split_entries.values()) != EXPECTED_N_VIDEOS:
    raise RuntimeError(
        "The recovered reference files do not sum to "
        f"{EXPECTED_N_VIDEOS} references."
    )

provenance = {
    "description": (
        "Recovered mirror of the INCLUDE-50 train/validation/test split "
        "used by the Dudu/Alves experimental pipeline."
    ),
    "repository": REFERENCE_SPLIT_REPOSITORY,
    "commit": REFERENCE_SPLIT_COMMIT,
    "counts": {
        split: len(entries)
        for split, entries in split_entries.items()
    },
    "sha256": {
        split: sha256_file(path)
        for split, path in required_split_files.items()
    },
    "local_files": {
        split: str(path)
        for split, path in required_split_files.items()
    },
}

PROVENANCE_PATH.write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nSplit provenance saved to:", PROVENANCE_PATH)


train: using existing local file (689 references)
val: using existing local file (77 references)
test: using existing local file (192 references)

Recovered reference split:
train: 689 references (expected 689) | sha256=ba36abcf06c5cfd2f7dc66babe8c0d46d407ebba50700ff2041b590dae2609f9
val: 77 references (expected 77) | sha256=60ed8603c22a5dbc06570cd3d9d466e561d071bf402dde1f5668e061e1ba34d0
test: 192 references (expected 192) | sha256=e8bd54ec642e2eea52f82730b4418b82268a4a34aa2df17e8bf66d54f2e1882e

Split provenance saved to: /Users/dani/Projetos/islr-subset/data/interim/include50_official/include50_reference_split_provenance.json


In [9]:
# Build strict lookup structures over candidate videos from the 50 classes.
selected_video_df = metadata.drop_duplicates("sequence_id").copy()

by_relpath = defaultdict(list)
by_basename = defaultdict(list)

for _, row in selected_video_df.iterrows():
    by_relpath[row["video_ref"]].append(row["sequence_id"])
    by_basename[row["video_basename_ref"]].append(row["sequence_id"])


def resolve_split_reference(reference: str) -> Tuple[Optional[str], str]:
    ref = norm_video_ref(reference)

    # 1) Exact normalized relative path
    exact = by_relpath.get(ref, [])
    if len(exact) == 1:
        return exact[0], "exact-relpath"
    if len(exact) > 1:
        return None, "ambiguous-exact-relpath"

    # 2) Unique suffix relation. This handles a split file that omits
    # the package prefix present in RAW_ROOT.
    suffix_matches = []
    for discovered_ref, seq_ids in by_relpath.items():
        if (
            discovered_ref.endswith("/" + ref)
            or ref.endswith("/" + discovered_ref)
        ):
            suffix_matches.extend(seq_ids)

    suffix_matches = sorted(set(suffix_matches))
    if len(suffix_matches) == 1:
        return suffix_matches[0], "unique-suffix"
    if len(suffix_matches) > 1:
        return None, "ambiguous-suffix"

    # 3) Unique basename only as a last resort.
    basename = ref.split("/")[-1]
    base_matches = by_basename.get(basename, [])
    if len(base_matches) == 1:
        return base_matches[0], "unique-basename"
    if len(base_matches) > 1:
        return None, "ambiguous-basename"

    return None, "not-found"


total_split_references = sum(len(v) for v in split_entries.values())
print("Total references in split files:", total_split_references)

if total_split_references != EXPECTED_N_VIDEOS:
    raise RuntimeError(
        f"Expected {EXPECTED_N_VIDEOS} total references across the "
        f"train/val/test files, found {total_split_references}."
    )

assignments = []
resolution_errors = []

for split, entries in split_entries.items():
    for reference in entries:
        sequence_id, mode = resolve_split_reference(reference)

        if sequence_id is None:
            resolution_errors.append({
                "split": split,
                "reference": reference,
                "problem": mode,
            })
        else:
            assignments.append({
                "split": split,
                "reference": reference,
                "sequence_id": sequence_id,
                "resolution_mode": mode,
            })

if resolution_errors:
    errors_df = pd.DataFrame(resolution_errors)
    display(errors_df)
    print("\nUNRESOLVED REFERENCE COUNT:", len(errors_df))
    raise RuntimeError(
        "One or more reference split videos are absent or ambiguous in the "
        "current raw data. The table above identifies the exact reference(s). "
        "Do not start MediaPipe extraction yet."
    )

assignment_df = pd.DataFrame(assignments)

# A sequence must occur only once across all split files.
duplicate_assignments = (
    assignment_df.groupby("sequence_id")["split"]
    .agg(list)
    .loc[lambda s: s.map(len) > 1]
)

if len(duplicate_assignments):
    display(duplicate_assignments)
    raise RuntimeError(
        "Some videos occur more than once across the reference split files."
    )

split_map = dict(zip(assignment_df["sequence_id"], assignment_df["split"]))

# IMPORTANT: the reference files, not every raw candidate video, define
# benchmark membership.
video_manifest = selected_video_df[
    selected_video_df["sequence_id"].isin(split_map)
].copy()

video_manifest["split"] = video_manifest["sequence_id"].map(split_map)

ignored_candidates = selected_video_df[
    ~selected_video_df["sequence_id"].isin(split_map)
][["sequence_id", "official_sign", "video_relpath"]].copy()

counts = video_manifest["split"].value_counts().to_dict()
n_train = int(counts.get("train", 0))
n_val = int(counts.get("val", 0))
n_test = int(counts.get("test", 0))

print("Resolved benchmark split counts:")
print("train:", n_train)
print("val:", n_val)
print("test:", n_test)
print("train + val:", n_train + n_val)
print("total:", len(video_manifest))
print("raw candidate videos ignored because they are not in the reference split:",
      len(ignored_candidates))

if n_train != EXPECTED_TRAIN:
    raise RuntimeError(
        f"Expected {EXPECTED_TRAIN} training videos, found {n_train}."
    )

if n_val != EXPECTED_VAL:
    raise RuntimeError(
        f"Expected {EXPECTED_VAL} validation videos, found {n_val}."
    )

if n_test != EXPECTED_TEST:
    raise RuntimeError(
        f"Expected {EXPECTED_TEST} test videos, found {n_test}."
    )

if n_train + n_val != EXPECTED_OFFICIAL_TRAIN_POOL:
    raise RuntimeError(
        f"Expected train + val = {EXPECTED_OFFICIAL_TRAIN_POOL}, "
        f"found {n_train + n_val}."
    )

if len(video_manifest) != EXPECTED_N_VIDEOS:
    raise RuntimeError(
        f"Expected {EXPECTED_N_VIDEOS} benchmark videos after split matching, "
        f"found {len(video_manifest)}."
    )

video_manifest.to_csv(MANIFEST_PATH, index=False)

print("OK: all 958 recovered reference split videos were matched exactly once.")
print("Manifest saved to:", MANIFEST_PATH)

if len(ignored_candidates):
    print("\nCandidate raw videos not used by the reference benchmark:")
    display(ignored_candidates.head(100))

display(
    video_manifest[
        ["sign_id", "official_sign", "category", "sign",
         "video_relpath", "split", "is_extra"]
    ].head(20)
)

Total references in split files: 958
Resolved benchmark split counts:
train: 689
val: 77
test: 192
train + val: 766
total: 958
raw candidate videos ignored because they are not in the reference split: 0
OK: all 958 recovered reference split videos were matched exactly once.
Manifest saved to: /Users/dani/Projetos/islr-subset/data/interim/include50_official/include50_official_manifest.csv


,sign_id,official_sign,category,sign,video_relpath,split,is_extra
0,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/Extra/MVI_3339.MOV,train,True
1,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/Extra/MVI_3420.MOV,train,True
2,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3335.MOV,test,False
3,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3336.MOV,train,False
4,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3337.MOV,train,False
5,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3338.MOV,test,False
6,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3419.MOV,test,False
7,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3421.MOV,val,False
8,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3422.MOV,train,False
9,0,Bank,Places,Bank,Places_4of4/Places/35. Bank/MVI_3423.MOV,train,False


In [10]:

# Per-class split inspection (diagnostic only)
split_by_class = (
    video_manifest.groupby(["sign_id", "official_sign", "split"])
    .size()
    .unstack(fill_value=0)
)

display(split_by_class)

print("\nClasses represented in each split:")
for split in ["train", "val", "test"]:
    if split in video_manifest["split"].unique():
        print(split, video_manifest.loc[video_manifest["split"] == split, "sign_id"].nunique())


,split,test,train,val
sign_id,official_sign,,,
0,Bank,6,14,2
1,Bird,1,22,2
2,Black,5,14,1
3,Boy,4,15,2
4,Brother,5,14,2
5,Car,4,14,2
6,Cell phone,4,9,1
7,Court,3,18,2
8,Cow,3,16,2



Classes represented in each split:
train 50
val 50
test 49


## MediaPipe Holistic extraction

In [11]:

# If MediaPipe is unavailable, install in the extraction environment:
# python -m pip install mediapipe==0.10.5 opencv-python-headless tqdm

import cv2
from tqdm.auto import tqdm
import mediapipe as mp

mp_holistic = mp.solutions.holistic

print("MediaPipe:", mp.__version__)
print("OpenCV:", cv2.__version__)


/Users/dani/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MediaPipe: 0.10.15
OpenCV: 4.11.0


In [12]:

def empty_landmarks(prefix: str, n_points: int) -> Dict[str, float]:
    row = {}
    for i in range(n_points):
        row[f"{prefix}_{i}_x"] = np.nan
        row[f"{prefix}_{i}_y"] = np.nan
        row[f"{prefix}_{i}_z"] = np.nan
    return row


def add_landmarks(row: Dict[str, float], prefix: str, landmarks, n_points: int):
    if landmarks is None:
        row.update(empty_landmarks(prefix, n_points))
        return True

    pts = landmarks.landmark

    for i in range(n_points):
        if i < len(pts):
            row[f"{prefix}_{i}_x"] = pts[i].x
            row[f"{prefix}_{i}_y"] = pts[i].y
            row[f"{prefix}_{i}_z"] = pts[i].z
        else:
            row[f"{prefix}_{i}_x"] = np.nan
            row[f"{prefix}_{i}_y"] = np.nan
            row[f"{prefix}_{i}_z"] = np.nan

    return False


def shard_path_for_sequence(sequence_id: str) -> Path:
    return SHARDS_DIR / f"{sequence_id}.csv"


def extract_video_to_rows(video_meta: pd.Series, holistic) -> pd.DataFrame:
    video_path = Path(video_meta["video_path"])
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    rows = []
    frame_id = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = holistic.process(rgb)

        row = {
            "sign_id": int(video_meta["sign_id"]),
            "category": video_meta["category"],
            "sign": video_meta["sign"],
            "official_sign": video_meta["official_sign"],
            "class_key": video_meta["class_key"],
            "sample_id": int(video_meta["sample_id"]),
            "video_name": video_meta["video_name"],
            "video_relpath": video_meta["video_relpath"],
            "sequence_id": video_meta["sequence_id"],
            "frame_id": frame_id,
            "split": video_meta["split"],
        }

        missing_hand_0 = add_landmarks(
            row, "hand_0", result.left_hand_landmarks, 21
        )
        missing_hand_1 = add_landmarks(
            row, "hand_1", result.right_hand_landmarks, 21
        )
        missing_pose = add_landmarks(
            row, "pose", result.pose_landmarks, 33
        )
        missing_face = add_landmarks(
            row, "face", result.face_landmarks, 468
        )

        row["missing_hand_0"] = bool(missing_hand_0)
        row["missing_hand_1"] = bool(missing_hand_1)
        row["missing_hand"] = bool(missing_hand_0 and missing_hand_1)
        row["missing_pose"] = bool(missing_pose)
        row["missing_face"] = bool(missing_face)

        rows.append(row)
        frame_id += 1

    cap.release()

    if not rows:
        raise RuntimeError(f"No frames could be read from video: {video_path}")

    return pd.DataFrame(rows)



## Extract videos

`RESUME=True` reuses only shards in the new `include50_official` directory.

For a quick smoke test, set `MAX_VIDEOS=5` in the configuration cell.  
Before producing the final CSV, return `MAX_VIDEOS=None` and ensure all 958 shards exist.


In [13]:

to_process = video_manifest.copy()

if MAX_VIDEOS is not None:
    to_process = to_process.head(MAX_VIDEOS).copy()

print("Videos in this run:", len(to_process))
print("RESUME:", RESUME)
print("Target full dataset:", EXPECTED_N_VIDEOS)

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    smooth_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.4,
    min_tracking_confidence=0.4,
) as holistic:

    for _, video_meta in tqdm(to_process.iterrows(), total=len(to_process)):
        out_path = shard_path_for_sequence(video_meta["sequence_id"])

        if RESUME and out_path.exists():
            continue

        try:
            shard_df = extract_video_to_rows(video_meta, holistic)
            shard_df.to_csv(out_path, index=False)

            # Remove stale error marker if this video now succeeded.
            error_path = out_path.with_suffix(".error.txt")
            if error_path.exists():
                error_path.unlink()

        except Exception as exc:
            error_path = out_path.with_suffix(".error.txt")
            error_path.write_text(str(exc), encoding="utf-8")
            print("Error while processing:", video_meta["video_relpath"])
            print(exc)

existing_shards = list(SHARDS_DIR.glob("*.csv"))
error_files = list(SHARDS_DIR.glob("*.error.txt"))

print("Existing shards:", len(existing_shards))
print("Extraction errors:", len(error_files))

if error_files:
    print("\nError files:")
    for p in error_files[:30]:
        print("-", p.name)


I0000 00:00:1786512776.162853 3943374 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Videos in this run: 958
RESUME: True
Target full dataset: 958


W0000 00:00:1786512776.273431 3944515 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  0%|          | 0/958 [00:00<?, ?it/s]W0000 00:00:1786512776.295859 3944515 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786512776.301852 3944514 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786512776.301981 3944515 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786512776.302817 3944520 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786512776.308297 3944520 inference_fee

Existing shards: 958
Extraction errors: 0


## Merge only after all 958 videos have been extracted

In [16]:

expected_sequence_ids = set(video_manifest["sequence_id"])
shard_paths = {
    p.stem: p
    for p in SHARDS_DIR.glob("*.csv")
}

missing_sequence_ids = sorted(expected_sequence_ids - set(shard_paths))
unexpected_sequence_ids = sorted(set(shard_paths) - expected_sequence_ids)

print("Expected videos:", len(expected_sequence_ids))
print("Available expected shards:", len(expected_sequence_ids & set(shard_paths)))
print("Missing expected shards:", len(missing_sequence_ids))
print("Unexpected shards in official directory:", len(unexpected_sequence_ids))

if unexpected_sequence_ids:
    raise RuntimeError(
        "Unexpected shard(s) exist in the official output directory. "
        "Remove them before merging:\n"
        + "\n".join(unexpected_sequence_ids[:30])
    )

if missing_sequence_ids:
    missing_info = (
        video_manifest[
            video_manifest["sequence_id"].isin(missing_sequence_ids)
        ][["sequence_id", "official_sign", "video_relpath", "split"]]
    )
    display(missing_info.head(100))

    raise RuntimeError(
        f"Cannot build final CSV: {len(missing_sequence_ids)} of the "
        f"{EXPECTED_N_VIDEOS} official videos have not been extracted yet."
    )

dfs = []

for sequence_id in tqdm(sorted(expected_sequence_ids)):
    dfs.append(pd.read_csv(shard_paths[sequence_id]))

final_df = pd.concat(dfs, ignore_index=True)

final_df = final_df.sort_values(
    ["sign_id", "sequence_id", "frame_id"]
).reset_index(drop=True)

final_df.to_csv(FINAL_CSV_PATH, index=False)

print("Final CSV saved to:", FINAL_CSV_PATH)
print("Rows:", len(final_df))
print("Videos:", final_df["sequence_id"].nunique())
print("Classes:", final_df["sign_id"].nunique())
print("\nSplit counts:")
print(
    final_df.drop_duplicates("sequence_id")["split"]
    .value_counts()
    .to_string()
)

display(final_df.head())


Expected videos: 958
Available expected shards: 958
Missing expected shards: 0
Unexpected shards in official directory: 0


100%|██████████| 958/958 [00:24<00:00, 39.49it/s]


Final CSV saved to: /Users/dani/Projetos/islr-subset/data/interim/include50_official/include50_official_mediapipe_with_split.csv
Rows: 60897
Videos: 958
Classes: 50

Split counts:
split
train    689
test     192
val       77


,sign_id,category,sign,official_sign,class_key,sample_id,video_name,video_relpath,sequence_id,frame_id,...,face_466_y,face_466_z,face_467_x,face_467_y,face_467_z,missing_hand_0,missing_hand_1,missing_hand,missing_pose,missing_face
0,0,Places,Bank,Bank,Places::Bank,17,MVI_3584.MOV,Places_4of4/Places/35. Bank/MVI_3584.MOV,07234b8fc3c4fbc72b0f21a7,0,...,0.306815,0.004833,0.523275,0.305777,0.005021,False,False,False,False,False
1,0,Places,Bank,Bank,Places::Bank,17,MVI_3584.MOV,Places_4of4/Places/35. Bank/MVI_3584.MOV,07234b8fc3c4fbc72b0f21a7,1,...,0.306177,0.005065,0.523129,0.305197,0.005233,False,False,False,False,False
2,0,Places,Bank,Bank,Places::Bank,17,MVI_3584.MOV,Places_4of4/Places/35. Bank/MVI_3584.MOV,07234b8fc3c4fbc72b0f21a7,2,...,0.305787,0.004825,0.523251,0.304778,0.004985,False,False,False,False,False
3,0,Places,Bank,Bank,Places::Bank,17,MVI_3584.MOV,Places_4of4/Places/35. Bank/MVI_3584.MOV,07234b8fc3c4fbc72b0f21a7,3,...,0.305727,0.004676,0.523258,0.304715,0.004832,False,False,False,False,False
4,0,Places,Bank,Bank,Places::Bank,17,MVI_3584.MOV,Places_4of4/Places/35. Bank/MVI_3584.MOV,07234b8fc3c4fbc72b0f21a7,4,...,0.305490,0.004666,0.523491,0.304492,0.004826,False,False,False,False,False


## Final integrity validation

In [17]:

validation_cols = [
    "sign_id",
    "category",
    "sign",
    "official_sign",
    "class_key",
    "sample_id",
    "video_name",
    "video_relpath",
    "sequence_id",
    "frame_id",
    "split",
    "missing_hand",
    "missing_hand_0",
    "missing_hand_1",
    "missing_face",
    "missing_pose",
]

df = pd.read_csv(FINAL_CSV_PATH, usecols=validation_cols)
videos = df.drop_duplicates("sequence_id").copy()

assert videos["sign_id"].nunique() == EXPECTED_N_CLASSES
assert videos["sequence_id"].nunique() == EXPECTED_N_VIDEOS
assert set(videos["split"].unique()) == {"train", "val", "test"}

counts = videos["split"].value_counts()

assert int(counts["train"]) == EXPECTED_TRAIN
assert int(counts["val"]) == EXPECTED_VAL
assert int(counts["test"]) == EXPECTED_TEST
assert int(counts["train"] + counts["val"]) == EXPECTED_OFFICIAL_TRAIN_POOL
assert int(counts.sum()) == EXPECTED_N_VIDEOS

print("FINAL VALIDATION PASSED")
print("=======================")
print("Classes:", videos["sign_id"].nunique())
print("Videos:", videos["sequence_id"].nunique())
print("Frames:", len(df))
print("\nSplit:")
print(counts.to_string())

print("\nVideos per class:")
print(
    videos.groupby(["sign_id", "official_sign"])["sequence_id"]
    .nunique()
    .to_string()
)

print("\nAverage missing rate per frame:")
print(
    df[
        [
            "missing_hand",
            "missing_hand_0",
            "missing_hand_1",
            "missing_face",
            "missing_pose",
        ]
    ].mean()
)

print("\nReady for:")
print("scripts/batch/run_include50_split.py")
print("\nUse this CSV:")
print(FINAL_CSV_PATH)


FINAL VALIDATION PASSED
Classes: 50
Videos: 958
Frames: 60897

Split:
split
train    689
test     192
val       77

Videos per class:
sign_id  official_sign
0        Bank             22
1        Bird             25
2        Black            20
3        Boy              21
4        Brother          21
5        Car              20
6        Cell phone       14
7        Court            23
8        Cow              21
9        Death            14
10       Dog              20
11       Election         14
12       Fall             15
13       Fan              15
14       Father           20
15       Girl             20
16       Good Morning     21
17       Hat              20
18       Hello            21
19       House            21
20       I                21
21       Monday           14
22       Paint            15
23       Pen              14
24       Priest           15
25       Red              20
26       Shoes            20
27       Shop             22
28       Summer           14
29

## Expected next command

After the final validation passes, point the INCLUDE fixed-split runner to:

`data/interim/include50_official/include50_official_mediapipe_with_split.csv`

The split provenance is saved alongside it at:

`data/interim/include50_official/include50_reference_split_provenance.json`

Do not reuse the previous `data/interim/include50/include50_mediapipe_with_split.csv`, because that file belongs to the provisional 929-video experiment.
